# CUAD QLoRA Fine-Tuning Benchmark
## Legal Contract QA: Base Model vs RAG Baseline vs QLoRA Fine-Tuned

This notebook fine-tunes **Llama-3.2-3B-Instruct** on legal compliance QA using **QLoRA** and
benchmarks it against a RAG-only baseline on the [CUAD dataset](https://huggingface.co/datasets/theatticusproject/cuad).

| System | Description |
|---|---|
| **Base Model** | Llama-3.2-3B-Instruct, no retrieval, no fine-tuning |
| **RAG Baseline** | FAISS retrieval + base Llama-3.2-3B-Instruct |
| **QLoRA Fine-tuned** | Same base model, fine-tuned on CUAD with LoRA |

**Eval metrics:** ROUGE-L · Token F1 · BERTScore  
**Experiment tracking:** MLflow (local)  
**Runtime:** Google Colab T4 GPU — run top-to-bottom sequentially

> ⚠️ **Before running:** Set runtime to **T4 GPU** via *Runtime → Change runtime type → T4 GPU*

---
## Section 1 — Install & Imports

Installs the full fine-tuning stack:
- **unsloth**: memory-optimised QLoRA kernels (2-5× faster than vanilla PEFT on T4)
- **trl / peft / bitsandbytes**: SFT trainer, LoRA, and 4-bit quantisation
- **faiss-cpu + sentence-transformers**: dense retrieval for the RAG baseline
- **evaluate / bert-score**: ROUGE-L, Token F1, BERTScore evaluation
- **mlflow**: experiment tracking (no external server needed in Colab)

In [2]:
# ── 1a  Install packages ──────────────────────────────────────────────────────
# unsloth must be installed first; it patches transformers at import time.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q transformers datasets peft bitsandbytes trl
!pip install -q faiss-cpu sentence-transformers
!pip install -q evaluate bert-score rouge-score
!pip install -q mlflow
print('All packages installed.')

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 103.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.4/418.4 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.8/224.8 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 63.8 MB/s 

In [3]:
# ── 1b  Imports & global seed ─────────────────────────────────────────────────
# unsloth MUST come first — it monkey-patches transformers/peft/trl at import
# time. Importing it after those libraries skips critical speed/memory optimisations.
import unsloth
from unsloth import FastLanguageModel

import os
import gc
import json
import random
import warnings

import numpy as np
import torch
import mlflow

from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

import faiss
from sentence_transformers import SentenceTransformer

import evaluate
from bert_score import score as bert_score_fn

from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# ── Global seed — applied to every RNG so experiments are reproducible ────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'GPU allocated   : {torch.cuda.memory_allocated() / 1e9:.2f} GB')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch version : 2.10.0+cu128
CUDA available  : True
GPU             : Tesla T4
GPU memory      : 15.6 GB
GPU allocated   : 0.01 GB


---
## Section 2 — Config Block

Single `CONFIG` dict that controls every hyperparameter in the notebook.
Edit values here — nothing else needs to change.

In [21]:
# ── 2  Master config ──────────────────────────────────────────────────────────
CONFIG = {
    # Model & data
    'model_name'                : 'unsloth/Llama-3.2-3B-Instruct',
    'dataset_name'              : 'cuad',
    'max_seq_length'            : 1024,

    # QLoRA adapter
    'lora_r'                    : 16,
    'lora_alpha'                : 32,
    'lora_dropout'              : 0.05,
    'target_modules'            : ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                                   'gate_proj', 'up_proj', 'down_proj'],

    # Training
    'train_split_size'          : 0.85,
    'num_train_epochs'          : 1,
    'per_device_train_batch_size': 1,
    'gradient_accumulation_steps': 8,   # effective batch = 2 * 4 = 8
    'learning_rate'             : 2e-4,
    'warmup_ratio'              : 0.03,
    'lr_scheduler_type'         : 'cosine',

    # Evaluation
    'test_sample_size'          : 100,

    # RAG
    'embedding_model'           : 'sentence-transformers/all-MiniLM-L6-v2',
    'top_k_retrieval'           : 3,

    # Tracking & deployment
    'mlflow_experiment'         : 'cuad-qlora-benchmark',
    'hf_model_repo'             : 'kumar-baibhav/llama3.2-3b-cuad-qlora',
}

print('CONFIG loaded:')
for k, v in CONFIG.items():
    print(f'  {k:<35}: {v}')

CONFIG loaded:
  model_name                         : unsloth/Llama-3.2-3B-Instruct
  dataset_name                       : cuad
  max_seq_length                     : 1024
  lora_r                             : 16
  lora_alpha                         : 32
  lora_dropout                       : 0.05
  target_modules                     : ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  train_split_size                   : 0.85
  num_train_epochs                   : 1
  per_device_train_batch_size        : 1
  gradient_accumulation_steps        : 8
  learning_rate                      : 0.0002
  warmup_ratio                       : 0.03
  lr_scheduler_type                  : cosine
  test_sample_size                   : 100
  embedding_model                    : sentence-transformers/all-MiniLM-L6-v2
  top_k_retrieval                    : 3
  mlflow_experiment                  : cuad-qlora-benchmark
  hf_model_repo                      : kumar-baibhav/llama

---
## Section 3 — Data Loading & Preprocessing

1. Load the CUAD dataset (510 full contracts, ~13 k QA pairs)
2. Filter out unanswerable questions (empty `answers["text"]`)
3. Format each row into the **Llama-3.2 instruction template** used for both training and inference
4. Create an 85/15 train-test split; sub-sample 100 rows for fast evaluation

In [5]:
# Upload CUADv1.json reliably via Colab's file picker (handles large files)
from google.colab import files

print('A file picker will appear — select CUADv1.json from your "data 3" folder …')
uploaded = files.upload()   # triggers browser file picker

import shutil
# Move to /content for easy access
for fname in uploaded:
    shutil.move(fname, '/content/CUADv1.json')
    print(f'Saved as /content/CUADv1.json  ({len(uploaded[fname]) / 1e6:.1f} MB)')


A file picker will appear — select CUADv1.json from your "data 3" folder …


Saving CUADv1.json to CUADv1.json
Saved as /content/CUADv1.json  (40.1 MB)


In [6]:
# ── 3a  Load CUAD from uploaded file ─────────────────────────────────────────
import json
from datasets import Dataset

print('Loading CUADv1.json …')
with open('/content/CUADv1.json') as f:
    cuad_json = json.load(f)
print(f"Version: {cuad_json['version']}  |  Articles: {len(cuad_json['data'])}")

rows = []
for article in cuad_json['data']:
    for para in article['paragraphs']:
        context = para['context']
        for qa in para['qas']:
            rows.append({
                'context' : context,
                'question': qa['question'],
                'answers' : {
                    'text'        : [a['text'] for a in qa['answers']],
                    'answer_start': [a['answer_start'] for a in qa['answers']],
                },
            })

train_data = Dataset.from_list(rows)
raw_dataset = {'train': train_data}

print(f'Columns   : {train_data.column_names}')
print(f'Train rows: {len(train_data)}')
sample = train_data[0]
print(f"\n  context (first 300 chars): {sample['context'][:300]}")
print(f"  question                 : {sample['question']}")
print(f"  answers                  : {sample['answers']}")


Loading CUADv1.json …
Version: aok_v1.0  |  Articles: 510
Columns   : ['context', 'question', 'answers']
Train rows: 20910

  context (first 300 chars): EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

        
  question                 : Highlight the parts (if any) of this contract related to "Document Name" that should be reviewed by a lawyer. Details: The name of the contract
  answers                  : {'answer_start': [44], 'text': ['DISTRIBUTOR AGREEMENT']}


In [7]:
# ── 3b  Filter empty answers ──────────────────────────────────────────────────
# Unanswerable questions (span not present in the contract) are excluded because
# supervised fine-tuning needs a concrete answer target.
def has_answer(example):
    return len(example['answers']['text']) > 0

filtered = train_data.filter(has_answer)
print(f'Rows after filtering empty answers: {len(filtered)}')
print(f'Removed (unanswerable)            : {len(train_data) - len(filtered)}')

# ── 3c  Instruction template (Llama-3.2 chat format) ─────────────────────────
SYSTEM_PROMPT = (
    'You are a legal contract analysis assistant. '
    'Answer questions about contracts accurately and concisely.'
)

def format_instruction(context: str, question: str, answer: str) -> str:
    """Build the full Llama-3.2 chat string used for training and as eval prompt."""
    return (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f'{SYSTEM_PROMPT}<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f'Contract Context:\n{context}\n\nQuestion: {question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
        f'{answer}<|eot_id|>'
    )

def preprocess(example):
    # Take first gold-standard answer span (CUAD may annotate multiple spans)
    answer = example['answers']['text'][0]
    example['formatted'] = format_instruction(
        example['context'], example['question'], answer
    )
    example['answer'] = answer   # plain text kept for metric computation
    return example

formatted_dataset = filtered.map(preprocess)
print(f'Formatted dataset size: {len(formatted_dataset)}')
print('\nSample formatted string (first 600 chars):')
print(formatted_dataset[0]['formatted'][:600])

Filter:   0%|          | 0/20910 [00:00<?, ? examples/s]

Rows after filtering empty answers: 6702
Removed (unanswerable)            : 14208


Map:   0%|          | 0/6702 [00:00<?, ? examples/s]

Formatted dataset size: 6702

Sample formatted string (first 600 chars):
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a legal contract analysis assistant. Answer questions about contracts accurately and concisely.<|eot_id|><|start_header_id|>user<|end_header_id|>

Contract Context:
EXHIBIT 10.6

                              DISTRIBUTOR AGREEMENT

         THIS  DISTRIBUTOR  AGREEMENT (the  "Agreement")  is made by and between Electric City Corp.,  a Delaware  corporation  ("Company")  and Electric City of Illinois LLC ("Distributor") this 7th day of September, 1999.

                                    RECITALS

         A. The  Company's  


In [9]:
# ── 3d  Train / test split ────────────────────────────────────────────────────
split = formatted_dataset.train_test_split(
    test_size=1 - CONFIG['train_split_size'],
    seed=SEED,
)
train_split = split['train']
test_split  = split['test']

# Sub-sample for fast eval; deterministic via SEED
eval_indices = random.sample(
    range(len(test_split)),
    min(CONFIG['test_sample_size'], len(test_split))
)
test_subset = test_split.select(eval_indices)

print(f'Train size  : {len(train_split)}')
print(f'Test size   : {len(test_split)}')
print(f'Eval subset : {len(test_subset)}')

# Extract raw fields used for inference and metric computation
eval_contexts  = list(test_subset['context'])
eval_questions = list(test_subset['question'])
eval_answers   = list(test_subset['answer'])

print('\nEval sample:')
print(f'  Q : {eval_questions[0]}')
print(f'  A : {eval_answers[0][:200]}')

Train size  : 5696
Test size   : 1006
Eval subset : 100

Eval sample:
  Q : Highlight the parts (if any) of this contract related to "License Grant" that should be reviewed by a lawyer. Details: Does the contract contain a license granted by one party to its counterparty?
  A : Subject to Section 3.2, a Licensor Party, on behalf of itself and the other members of the Licensor Group, and solely to the extent the Licensor Party or another member of the Licensor Group has the r


---
## Section 4 — RAG Baseline

Build a **retrieval-augmented generation** pipeline:
- **4a** — Encode all training contexts into a FAISS flat inner-product index
- **4b** — Implement a `retrieve(query, k)` helper
- **4c** — Load Llama-3.2-3B-Instruct in 4-bit (base weights, no LoRA) + `generate_answer`
- **4d** — Run inference: base-only pass & RAG pass on the 100-row eval set
- **4e** — Compute ROUGE-L, Token F1, BERTScore for both systems

In [10]:
# ── 4a  Build FAISS retrieval index ──────────────────────────────────────────
# We embed every training-split context with a lightweight sentence-transformer.
# IndexFlatIP (inner product) on L2-normalised vectors == cosine similarity.
# Exact search is fine at this scale (~10 k vectors).
print('Loading embedding model …')
embedder = SentenceTransformer(CONFIG['embedding_model'])

train_contexts_list = list(train_split['context'])
print(f'Encoding {len(train_contexts_list)} training contexts …')

context_embeddings = embedder.encode(
    train_contexts_list,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # pre-normalise → dot product == cosine sim
)

dim = context_embeddings.shape[1]
print(f'Embedding shape: {context_embeddings.shape}')

faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(context_embeddings.astype('float32'))
print(f'FAISS index built: {faiss_index.ntotal} vectors, dim={dim}')
print(f'GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Loading embedding model …


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 5696 training contexts …


Batches:   0%|          | 0/45 [00:00<?, ?it/s]

Embedding shape: (5696, 384)
FAISS index built: 5696 vectors, dim=384
GPU memory allocated: 0.10 GB


In [11]:
# ── 4b  Retrieval helper ──────────────────────────────────────────────────────
def retrieve(query: str, k: int = CONFIG['top_k_retrieval']) -> list:
    """
    Encode the query, L2-normalise, search the FAISS index,
    return the top-k training context strings.
    """
    q_emb = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype('float32')
    faiss.normalize_L2(q_emb)   # safety guard — already normalised above
    _, indices = faiss_index.search(q_emb, k)
    return [
        train_contexts_list[i]
        for i in indices[0]
        if 0 <= i < len(train_contexts_list)
    ]

# Quick smoke-test
sample_ctxs = retrieve(eval_questions[0])
print(f'Retrieved {len(sample_ctxs)} contexts for:')
print(f'  Q: {eval_questions[0][:80]}…')
print(f'  First result (first 300 chars): {sample_ctxs[0][:300]}')

Retrieved 3 contexts for:
  Q: Highlight the parts (if any) of this contract related to "License Grant" that sh…
  First result (first 300 chars): Exhibit 10.4 CONSULTING AGREEMENT This Consulting Agreement ("Agreement") is made and entered into as of May 1, 2019 ("Effective Date") by and between Driven Deliveries, Inc. ("Company"), a Nevada corporation, and TruckThat LLC ("Consultant"). Company and Consultant shall sometimes be referred to he


In [12]:
# ── 4c  Load base model (4-bit, no LoRA) ─────────────────────────────────────
# unsloth's FastLanguageModel handles 4-bit quantisation, flash-attention,
# and RoPE scaling automatically. No LoRA adapters are added here.
print('Loading base Llama-3.2-3B-Instruct (4-bit) …')
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    dtype=None,          # auto: bfloat16 on Ampere+, float16 on T4
    load_in_4bit=True,
)
# Switch to inference mode — disables dropout, enables KV-cache
FastLanguageModel.for_inference(base_model)

total_params = sum(p.numel() for p in base_model.parameters())
print(f'Parameters  : {total_params / 1e9:.2f} B')
print(f'GPU memory  : {torch.cuda.memory_allocated() / 1e9:.2f} GB')


def generate_answer(prompt: str, model, tokenizer, max_new_tokens: int = 256) -> str:
    """
    Greedy-decode a single prompt and return only the newly generated tokens.
    - torch.no_grad() avoids storing gradients → saves ~30% GPU memory.
    - do_sample=False gives deterministic, reproducible outputs.
    """
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=CONFIG['max_seq_length'],
    ).to(model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Strip the prompt tokens; decode only the generated portion
    new_tokens = output_ids[0][input_len:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


print('generate_answer helper ready.')

Loading base Llama-3.2-3B-Instruct (4-bit) …
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.35G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Parameters  : 1.84 B
GPU memory  : 2.49 GB
generate_answer helper ready.


In [13]:
# ── 4d  Inference: Base model & RAG baseline ─────────────────────────────────
# Base model: question only — tests the model's parametric (memorised) knowledge.
# RAG:        top-k contexts prepended — tests whether retrieved evidence helps.

def make_prompt_base(question: str) -> str:
    """No retrieved context; pure parametric recall."""
    return (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f'{SYSTEM_PROMPT}<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f'Question: {question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )


def make_prompt_rag(question: str, contexts: list) -> str:
    """Prepend retrieved contract snippets so the model can ground its answer."""
    ctx_block = '\n\n---\n\n'.join(contexts)
    return (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f'{SYSTEM_PROMPT}<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f'Relevant Contract Excerpts:\n{ctx_block}\n\n'
        f'Question: {question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )


# ── Base model inference ──────────────────────────────────────────────────────
print('Running BASE MODEL inference …')
base_preds = []
for q in tqdm(eval_questions, desc='Base model'):
    pred = generate_answer(make_prompt_base(q), base_model, base_tokenizer)
    base_preds.append(pred if pred else '')   # edge-case: empty → empty string

# ── RAG inference ─────────────────────────────────────────────────────────────
print('\nRunning RAG BASELINE inference …')
rag_preds = []
for q in tqdm(eval_questions, desc='RAG baseline'):
    ctxs = retrieve(q)
    pred = generate_answer(make_prompt_rag(q, ctxs), base_model, base_tokenizer)
    rag_preds.append(pred if pred else '')

print(f'\nSample base pred : {base_preds[0][:200]}')
print(f'Sample RAG pred  : {rag_preds[0][:200]}')
print(f'Gold answer      : {eval_answers[0][:200]}')
print(f'GPU memory       : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Running BASE MODEL inference …


Base model:   0%|          | 0/100 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


Running RAG BASELINE inference …


RAG baseline:   0%|          | 0/100 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


Sample base pred : I'm ready to assist. Please provide the contract text, and I'll highlight the relevant parts related to a "License Grant" that should be reviewed by a lawyer.
Sample RAG pred  : , including but not limited to any confidential documents, trade secrets, or other proprietary information, without the prior written consent of an authorized representative of Company. 2.3. Return of
Gold answer      : Subject to Section 3.2, a Licensor Party, on behalf of itself and the other members of the Licensor Group, and solely to the extent the Licensor Party or another member of the Licensor Group has the r
GPU memory       : 2.70 GB


In [14]:
# ── 4e  Evaluation helpers & base / RAG results ───────────────────────────────
rouge_metric = evaluate.load('rouge')


def compute_rouge_l(preds: list, refs: list) -> float:
    """ROUGE-L F1 averaged over all prediction-reference pairs."""
    # Replace empty preds with a single space so the scorer doesn't crash
    safe_preds = [p if p.strip() else ' ' for p in preds]
    result = rouge_metric.compute(
        predictions=safe_preds, references=refs, use_stemmer=True
    )
    return round(result['rougeL'], 4)


def compute_token_f1(preds: list, refs: list) -> dict:
    """
    Token-level precision / recall / F1 using whitespace tokenisation.
    Mirrors the SQuAD evaluation script; treats prediction & reference
    as bags-of-words and computes overlap.
    """
    precisions, recalls, f1s = [], [], []
    for pred, ref in zip(preds, refs):
        pred_toks = set(pred.lower().split())
        ref_toks  = set(ref.lower().split())
        if not pred_toks and not ref_toks:
            precisions.append(1.0); recalls.append(1.0); f1s.append(1.0)
            continue
        if not pred_toks or not ref_toks:
            precisions.append(0.0); recalls.append(0.0); f1s.append(0.0)
            continue
        common = pred_toks & ref_toks
        p = len(common) / len(pred_toks)
        r = len(common) / len(ref_toks)
        f = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
        precisions.append(p); recalls.append(r); f1s.append(f)
    return {
        'precision': round(float(np.mean(precisions)), 4),
        'recall':    round(float(np.mean(recalls)),    4),
        'f1':        round(float(np.mean(f1s)),        4),
    }


def compute_bertscore(preds: list, refs: list) -> float:
    """
    BERTScore F1 using distilbert-base-uncased (smaller/faster on T4 than
    the default roberta-large). Falls back to 0.0 on any failure.
    """
    try:
        safe_preds = [p if p.strip() else ' ' for p in preds]
        _, _, F = bert_score_fn(
            safe_preds, refs,
            model_type='distilbert-base-uncased',
            lang='en',
            verbose=False,
        )
        return round(float(F.mean().item()), 4)
    except Exception as exc:
        print(f'BERTScore failed: {exc}')
        return 0.0


# ── Compute metrics ───────────────────────────────────────────────────────────
print('Evaluating base model …')
base_rouge  = compute_rouge_l(base_preds, eval_answers)
base_tf1    = compute_token_f1(base_preds, eval_answers)
base_bscore = compute_bertscore(base_preds, eval_answers)

print('Evaluating RAG baseline …')
rag_rouge   = compute_rouge_l(rag_preds, eval_answers)
rag_tf1     = compute_token_f1(rag_preds, eval_answers)
rag_bscore  = compute_bertscore(rag_preds, eval_answers)

sep = '-' * 58
print(f'\n{sep}')
print(f'{"System":<22} {"ROUGE-L":>10} {"Token F1":>10} {"BERTScore":>12}')
print(sep)
print(f'{"Base Model":<22} {base_rouge:>10.4f} {base_tf1["f1"]:>10.4f} {base_bscore:>12.4f}')
print(f'{"RAG Baseline":<22} {rag_rouge:>10.4f} {rag_tf1["f1"]:>10.4f} {rag_bscore:>12.4f}')
print(sep)

Evaluating base model …


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Evaluating RAG baseline …


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



----------------------------------------------------------
System                    ROUGE-L   Token F1    BERTScore
----------------------------------------------------------
Base Model                 0.0881     0.1099       0.6878
RAG Baseline               0.0711     0.1054       0.6435
----------------------------------------------------------


In [23]:
# Free RAG components — not needed for fine-tuning
del embedder, faiss_index, context_embeddings, base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()
print(f'GPU free: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB')


NameError: name 'embedder' is not defined

---
## Section 5 — QLoRA Fine-Tuning

Fine-tune Llama-3.2-3B-Instruct on CUAD with **QLoRA**:
- **5a** — Reload model with LoRA adapters (rank-16, ~20 M trainable params)
- **5b** — Configure `SFTTrainer` with the formatted instruction dataset
- **5c** — Train for 3 epochs; print final loss
- **5d** — Run inference on the same 100-row eval set

> ⏱️ Training takes ~45–60 minutes on a T4 16 GB with the default config.

In [27]:
# ── 5a  Load model with QLoRA adapters ───────────────────────────────────────
# LoRA inserts small rank-r trainable matrices alongside frozen weight matrices.
# With r=16 we go from ~3 B trainable params to ~20 M — a 150× reduction —
# while retaining most of the base model's knowledge.
print('Loading model with QLoRA …')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG['model_name'],
    max_seq_length=CONFIG['max_seq_length'],
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG['lora_r'],
    target_modules=CONFIG['target_modules'],
    lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    use_gradient_checkpointing='unsloth',  # unsloth's smart checkpointing saves VRAM
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters    : {total_params / 1e6:.1f} M')
print(f'Trainable parameters: {trainable_params / 1e6:.1f} M  '
      f'({100 * trainable_params / total_params:.2f}%)')
print(f'GPU memory          : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Loading model with QLoRA …
==((====))==  Unsloth 2026.4.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Total parameters    : 1865.5 M
Trainable parameters: 24.3 M  (1.30%)
GPU memory          : 5.02 GB


In [28]:
# ── 5b  SFTTrainer configuration ─────────────────────────────────────────────
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'   # prevent fork-related crashes

# Pre-tokenize with a single process — bypasses unsloth's internal map() call
# which ignores dataset_num_proc and uses cpu_count() causing subprocess OOM
print('Pre-tokenizing training data (num_proc=1) …')

def tokenize_batch(examples):
    tokenized = tokenizer(
        examples['formatted'],
        truncation=True,
        max_length=CONFIG['max_seq_length'],
        padding=False,
    )
    tokenized['labels'] = tokenized['input_ids'].copy()
    return tokenized

tokenized_train = train_split.map(
    tokenize_batch,
    batched=True,
    batch_size=32,
    num_proc=1,
    remove_columns=train_split.column_names,
    desc='Tokenizing',
)
print(f'Tokenized: {len(tokenized_train)} rows | columns: {tokenized_train.column_names}')

# Compute warmup steps
steps_per_epoch = len(tokenized_train) // (
    CONFIG['per_device_train_batch_size'] * CONFIG['gradient_accumulation_steps']
)
total_steps  = steps_per_epoch * CONFIG['num_train_epochs']
warmup_steps = max(1, int(total_steps * CONFIG['warmup_ratio']))
print(f'Steps per epoch: {steps_per_epoch} | Total: {total_steps} | Warmup: {warmup_steps}')

training_args = TrainingArguments(
    output_dir='./cuad-qlora-checkpoints',
    num_train_epochs=CONFIG['num_train_epochs'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    learning_rate=CONFIG['learning_rate'],
    warmup_steps=warmup_steps,
    lr_scheduler_type=CONFIG['lr_scheduler_type'],
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='epoch',
    report_to='none',
    seed=SEED,
    optim='adamw_8bit',
    weight_decay=0.01,
    max_grad_norm=1.0,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized_train,   # already tokenized — skips internal map()
    max_seq_length=CONFIG['max_seq_length'],
    dataset_num_proc=1,
    packing=False,
    args=training_args,
)
print('SFTTrainer ready.')


Pre-tokenizing training data (num_proc=1) …


Tokenizing (num_proc=1):   0%|          | 0/5696 [00:00<?, ? examples/s]

Tokenized: 5696 rows | columns: ['input_ids', 'attention_mask', 'labels']
Steps per epoch: 712 | Total: 712 | Warmup: 21
SFTTrainer ready.


In [29]:
# ── 5c  Fine-tune ─────────────────────────────────────────────────────────────
# Training ~3 epochs on CUAD takes ~45–60 min on T4 with these settings.
# Typical loss trajectory: starts ~2.x, converges to ~0.5–1.0 on legal text.
print('Starting QLoRA fine-tuning …')
train_result = trainer.train()

print('\n' + '=' * 50)
print('Training complete!')
print(f'  Final training loss : {train_result.training_loss:.4f}')
print(f'  Total steps         : {train_result.global_step}')
runtime = train_result.metrics.get('train_runtime', 0)
print(f'  Runtime             : {runtime:.0f} s  ({runtime / 60:.1f} min)')
print(f'  GPU memory          : {torch.cuda.memory_allocated() / 1e9:.2f} GB')
print('=' * 50)

Starting QLoRA fine-tuning …


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,696 | Num Epochs = 1 | Total steps = 712
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
10,1.966035
20,1.765599
30,1.732387
40,1.602797
50,1.574244
60,1.555138
70,1.481800
80,1.415537
90,1.441730
100,1.353332



Training complete!
  Final training loss : 0.7564
  Total steps         : 712
  Runtime             : 9799 s  (163.3 min)
  GPU memory          : 3.38 GB


In [26]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

for var in ['base_model', 'base_tokenizer', 'embedder', 'faiss_index', 'context_embeddings']:
    if var in dir():
        exec(f'del {var}')

gc.collect()
torch.cuda.empty_cache()
print(f'GPU free : {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated()) / 1e9:.2f} GB')
print(f'GPU used : {torch.cuda.memory_allocated() / 1e9:.2f} GB')


GPU free : 13.11 GB
GPU used : 2.53 GB


In [30]:
# ── 5d  Fine-tuned model inference ───────────────────────────────────────────
# After fine-tuning, the model learned to extract answers from provided context.
# We pass the SAME context+question format used during training — no retrieval.
# This tests whether the model generalised to unseen contracts.
FastLanguageModel.for_inference(model)   # re-enable KV-cache, disable dropout


def make_prompt_finetuned(context: str, question: str) -> str:
    """Matches the training format exactly, but omits the assistant answer token."""
    return (
        '<|begin_of_text|>'
        '<|start_header_id|>system<|end_header_id|>\n\n'
        f'{SYSTEM_PROMPT}<|eot_id|>'
        '<|start_header_id|>user<|end_header_id|>\n\n'
        f'Contract Context:\n{context}\n\nQuestion: {question}<|eot_id|>'
        '<|start_header_id|>assistant<|end_header_id|>\n\n'
    )


print('Running QLoRA fine-tuned model inference …')
ft_preds = []
for ctx, q in tqdm(
    zip(eval_contexts, eval_questions),
    total=len(eval_questions),
    desc='Fine-tuned model',
):
    pred = generate_answer(make_prompt_finetuned(ctx, q), model, tokenizer)
    ft_preds.append(pred if pred else '')

print(f'\nSample fine-tuned pred : {ft_preds[0][:200]}')
print(f'Gold answer            : {eval_answers[0][:200]}')
print(f'GPU memory             : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

# Free training state before evaluation; gradients are no longer needed
gc.collect()
torch.cuda.empty_cache()
print(f'GPU memory after cache clear: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

Running QLoRA fine-tuned model inference …


Fine-tuned model:   0%|          | 0/100 [00:00<?, ?it/s]

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_


Sample fine-tuned pred : forth herein. NOW THEREFORE, in consideration of the foregoing premises and the mutual covenants set forth below, the Parties hereto agree as follows:

1

Source: UNITED TECHNOLOGIES CORPORATION, 10-1
Gold answer            : Subject to Section 3.2, a Licensor Party, on behalf of itself and the other members of the Licensor Group, and solely to the extent the Licensor Party or another member of the Licensor Group has the r
GPU memory             : 3.60 GB
GPU memory after cache clear: 3.60 GB


In [32]:
print('Running QLoRA fine-tuned model inference …')
ft_preds = []
for ctx, q in tqdm(
    zip(eval_contexts, eval_questions),
    total=len(eval_questions),
    desc='Fine-tuned model',
):
    prompt = make_prompt_finetuned(ctx, q)
    pred   = generate_answer(prompt, model, tokenizer, max_new_tokens=64)  # was 256
    ft_preds.append(pred if pred else '')


Running QLoRA fine-tuned model inference …


Fine-tuned model:   0%|          | 0/100 [00:00<?, ?it/s]

Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=64) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_gene

---
## Section 6 — Evaluation & MLflow Logging

- **6a** — Compute ROUGE-L, Token F1, BERTScore for the fine-tuned model
- **6b** — Log all three systems to MLflow (3 separate runs)
- **6c** — Print the full comparison table with delta improvements

In [33]:
# ── 6a  Evaluate fine-tuned model ─────────────────────────────────────────────
print('Evaluating QLoRA fine-tuned model …')
ft_rouge  = compute_rouge_l(ft_preds, eval_answers)
ft_tf1    = compute_token_f1(ft_preds, eval_answers)
ft_bscore = compute_bertscore(ft_preds, eval_answers)

print(f'  ROUGE-L      : {ft_rouge}')
print(f'  Token F1     : {ft_tf1}')
print(f'  BERTScore F1 : {ft_bscore}')

Evaluating QLoRA fine-tuned model …


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  ROUGE-L      : 0.11
  Token F1     : {'precision': 0.1465, 'recall': 0.1718, 'f1': 0.1374}
  BERTScore F1 : 0.6955


In [34]:
# ── 6b  Log all three systems to MLflow ───────────────────────────────────────
# One MLflow run per system makes it easy to compare in the MLflow UI.
# Every CONFIG entry is logged as a parameter so runs are fully reproducible.
mlflow.set_experiment(CONFIG['mlflow_experiment'])

systems = {
    'base_model': {
        'rouge_l':      base_rouge,
        'token_f1':     base_tf1['f1'],
        'bertscore_f1': base_bscore,
    },
    'rag_baseline': {
        'rouge_l':      rag_rouge,
        'token_f1':     rag_tf1['f1'],
        'bertscore_f1': rag_bscore,
    },
    'qlora_finetuned': {
        'rouge_l':      ft_rouge,
        'token_f1':     ft_tf1['f1'],
        'bertscore_f1': ft_bscore,
    },
}

for run_name, metrics in systems.items():
    with mlflow.start_run(run_name=run_name):
        # Log every config key as a param (stringify for MLflow compatibility)
        for k, v in CONFIG.items():
            mlflow.log_param(k, str(v))
        mlflow.log_param('seed', SEED)
        mlflow.log_param('eval_sample_size', len(eval_questions))
        for metric_name, value in metrics.items():
            mlflow.log_metric(metric_name, value)
    print(f'Logged: {run_name:20s} → {metrics}')

print(f'\nMLflow experiment : {CONFIG["mlflow_experiment"]}')
print('To launch the MLflow UI in Colab, run:')
print('  !mlflow ui --port 5000 &')
print('  from google.colab.output import eval_js; print(eval_js("google.colab.kernel.proxyPort(5000)"))')

2026/04/12 05:10:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/12 05:10:50 INFO mlflow.store.db.utils: Updating database tables
2026/04/12 05:10:52 INFO mlflow.tracking.fluent: Experiment with name 'cuad-qlora-benchmark' does not exist. Creating a new experiment.


Logged: base_model           → {'rouge_l': np.float64(0.0881), 'token_f1': 0.1099, 'bertscore_f1': 0.6878}
Logged: rag_baseline         → {'rouge_l': np.float64(0.0711), 'token_f1': 0.1054, 'bertscore_f1': 0.6435}
Logged: qlora_finetuned      → {'rouge_l': np.float64(0.11), 'token_f1': 0.1374, 'bertscore_f1': 0.6955}

MLflow experiment : cuad-qlora-benchmark
To launch the MLflow UI in Colab, run:
  !mlflow ui --port 5000 &
  from google.colab.output import eval_js; print(eval_js("google.colab.kernel.proxyPort(5000)"))


In [35]:
# ── 6c  Final comparison table ────────────────────────────────────────────────
results = [
    ('Base Model',       base_rouge, base_tf1['f1'], base_bscore),
    ('RAG Baseline',     rag_rouge,  rag_tf1['f1'],  rag_bscore),
    ('QLoRA Fine-tuned', ft_rouge,   ft_tf1['f1'],   ft_bscore),
]

header = f'{"System":<22} {"ROUGE-L":>10} {"Token F1":>10} {"BERTScore F1":>14}'
border = '=' * len(header)
thin   = '-' * len(header)

print(f'\n{border}')
print(header)
print(thin)
for name, r, f, b in results:
    print(f'{name:<22} {r:>10.4f} {f:>10.4f} {b:>14.4f}')
print(border)

# Delta: QLoRA fine-tuned vs RAG baseline
delta_r = ft_rouge       - rag_rouge
delta_f = ft_tf1['f1']   - rag_tf1['f1']
delta_b = ft_bscore      - rag_bscore

print('\nImprovement of QLoRA over RAG Baseline:')
print(f'  ΔROUGE-L      : {delta_r:+.4f}  ({"↑ improvement" if delta_r >= 0 else "↓ regression"})')
print(f'  ΔToken F1     : {delta_f:+.4f}  ({"↑ improvement" if delta_f >= 0 else "↓ regression"})')
print(f'  ΔBERTScore F1 : {delta_b:+.4f}  ({"↑ improvement" if delta_b >= 0 else "↓ regression"})')


System                    ROUGE-L   Token F1   BERTScore F1
-----------------------------------------------------------
Base Model                 0.0881     0.1099         0.6878
RAG Baseline               0.0711     0.1054         0.6435
QLoRA Fine-tuned           0.1100     0.1374         0.6955

Improvement of QLoRA over RAG Baseline:
  ΔROUGE-L      : +0.0389  (↑ improvement)
  ΔToken F1     : +0.0320  (↑ improvement)
  ΔBERTScore F1 : +0.0520  (↑ improvement)


---
## Section 7 — Save & Push to HuggingFace Hub

1. Save LoRA adapter weights and tokenizer locally
2. Auto-generate a model card (`README.md`) with training config and results
3. Print push instructions for `huggingface-cli` + `push_to_hub`

In [36]:
# ── 7a  Save LoRA adapter & tokenizer ─────────────────────────────────────────
# Only the LoRA delta weights are saved (~80 MB), NOT the full base model.
# Users load the base model separately and merge adapters at inference time.
ADAPTER_DIR = 'cuad-qlora-adapter'

print(f'Saving LoRA adapter to "{ADAPTER_DIR}" …')
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print('\nSaved files:')
for fname in sorted(os.listdir(ADAPTER_DIR)):
    size_mb = os.path.getsize(os.path.join(ADAPTER_DIR, fname)) / 1e6
    print(f'  {fname:<45} {size_mb:6.1f} MB')

Saving LoRA adapter to "cuad-qlora-adapter" …

Saved files:
  README.md                                        0.0 MB
  adapter_config.json                              0.0 MB
  adapter_model.safetensors                       97.3 MB
  chat_template.jinja                              0.0 MB
  tokenizer.json                                  17.2 MB
  tokenizer_config.json                            0.0 MB


In [37]:
# ── 7b  Generate model card (README.md) ───────────────────────────────────────
# A well-formed model card helps other users find, understand, and reproduce
# this model. The card is written to the adapter directory so it uploads
# automatically when push_to_hub() is called.

model_card_lines = [
    '---',
    'language: en',
    'license: llama3.2',
    f'base_model: {CONFIG["model_name"]}',
    'tags:',
    '  - legal',
    '  - contract-analysis',
    '  - qlora',
    '  - peft',
    '  - llama',
    '  - cuad',
    'datasets:',
    '  - theatticusproject/cuad',
    '---',
    '',
    f'# {CONFIG["hf_model_repo"]}',
    '',
    'QLoRA fine-tuned adapter for **legal contract question answering** on the',
    '[CUAD dataset](https://huggingface.co/datasets/theatticusproject/cuad).',
    '',
    '## Training Configuration',
    '',
    '| Parameter | Value |',
    '|---|---|',
    f'| Base model | `{CONFIG["model_name"]}` |',
    f'| LoRA rank (r) | {CONFIG["lora_r"]} |',
    f'| LoRA alpha | {CONFIG["lora_alpha"]} |',
    f'| LoRA dropout | {CONFIG["lora_dropout"]} |',
    f'| Epochs | {CONFIG["num_train_epochs"]} |',
    f'| Effective batch size | {CONFIG["per_device_train_batch_size"] * CONFIG["gradient_accumulation_steps"]} |',
    f'| Learning rate | {CONFIG["learning_rate"]} |',
    f'| LR scheduler | {CONFIG["lr_scheduler_type"]} |',
    f'| Max sequence length | {CONFIG["max_seq_length"]} |',
    '| Quantisation | 4-bit NF4 (QLoRA) |',
    '',
    f'## Evaluation Results (CUAD test set, n={CONFIG["test_sample_size"]})',
    '',
    '| System | ROUGE-L | Token F1 | BERTScore F1 |',
    '|---|---|---|---|',
    f'| Base Model | {base_rouge:.4f} | {base_tf1["f1"]:.4f} | {base_bscore:.4f} |',
    f'| RAG Baseline | {rag_rouge:.4f} | {rag_tf1["f1"]:.4f} | {rag_bscore:.4f} |',
    f'| **QLoRA Fine-tuned** | **{ft_rouge:.4f}** | **{ft_tf1["f1"]:.4f}** | **{ft_bscore:.4f}** |',
    '',
    '## Load & Inference',
    '',
    '```python',
    'from peft import PeftModel',
    'from transformers import AutoModelForCausalLM, AutoTokenizer',
    '',
    f'base = AutoModelForCausalLM.from_pretrained(',
    f'    "{CONFIG["model_name"]}",',
    '    load_in_4bit=True,',
    '    device_map="auto",',
    ')',
    f'tokenizer = AutoTokenizer.from_pretrained("{CONFIG["hf_model_repo"]}")',
    f'model = PeftModel.from_pretrained(base, "{CONFIG["hf_model_repo"]}")',
    '',
    'prompt = (',
    '    "<|begin_of_text|>"',
    '    "<|start_header_id|>system<|end_header_id|>\\n\\n"',
    '    "You are a legal contract analysis assistant. "',
    '    "Answer questions about contracts accurately and concisely.<|eot_id|>"',
    '    "<|start_header_id|>user<|end_header_id|>\\n\\n"',
    '    "Contract Context:\\n{your_contract_text}\\n\\n"',
    '    "Question: {your_question}<|eot_id|>"',
    '    "<|start_header_id|>assistant<|end_header_id|>\\n\\n"',
    ')',
    'inputs = tokenizer(prompt, return_tensors="pt").to(model.device)',
    'out = model.generate(**inputs, max_new_tokens=256, do_sample=False)',
    'print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))',
    '```',
]

model_card_text = '\n'.join(model_card_lines)
card_path = os.path.join(ADAPTER_DIR, 'README.md')
with open(card_path, 'w', encoding='utf-8') as fh:
    fh.write(model_card_text)

print(f'Model card written: {card_path}')
print(f'\nPreview (first 500 chars):\n{model_card_text[:500]}')

Model card written: cuad-qlora-adapter/README.md

Preview (first 500 chars):
---
language: en
license: llama3.2
base_model: unsloth/Llama-3.2-3B-Instruct
tags:
  - legal
  - contract-analysis
  - qlora
  - peft
  - llama
  - cuad
datasets:
  - theatticusproject/cuad
---

# kumar-baibhav/llama3.2-3b-cuad-qlora

QLoRA fine-tuned adapter for **legal contract question answering** on the
[CUAD dataset](https://huggingface.co/datasets/theatticusproject/cuad).

## Training Configuration

| Parameter | Value |
|---|---|
| Base model | `unsloth/Llama-3.2-3B-Instruct` |
| LoRA ran


In [38]:
import shutil
from google.colab import files

# Zip the adapter folder
shutil.make_archive('cuad-qlora-adapter', 'zip', '/content/cuad-qlora-adapter')

# Download to your local machine
files.download('cuad-qlora-adapter.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [39]:
# ── 7c  Push to HuggingFace Hub ───────────────────────────────────────────────
# The adapter directory (~80 MB) contains only LoRA delta weights + tokenizer.
# Users pull the 4-bit base model separately and apply the adapter at runtime.

print('=' * 60)
print('PUSH TO HUGGINGFACE HUB')
print('=' * 60)
print()
print('Step 1 — Authenticate (run once):')
print('    !huggingface-cli login')
print()
print('Step 2 — Push adapter + tokenizer:')
print(f'    model.push_to_hub("{CONFIG["hf_model_repo"]}", safe_serialization=True)')
print(f'    tokenizer.push_to_hub("{CONFIG["hf_model_repo"]}")')
print()
print('Step 3 (optional) — Merge adapters into base for standalone model:')
print('    merged = model.merge_and_unload()')
print(f'    merged.push_to_hub("{CONFIG["hf_model_repo"]}-merged")')
print()
print('-' * 60)
print('Programmatic push (uncomment after huggingface-cli login):')
print('-' * 60)

# Uncomment the lines below after running: !huggingface-cli login
# model.push_to_hub(CONFIG['hf_model_repo'], safe_serialization=True)
# tokenizer.push_to_hub(CONFIG['hf_model_repo'])

print(f'\nAdapter directory  : {ADAPTER_DIR}/')
print(f'Target HF repo     : https://huggingface.co/{CONFIG["hf_model_repo"]}')
print(f'GPU memory (final) : {torch.cuda.memory_allocated() / 1e9:.2f} GB')

print()
print('=' * 60)
print('ALL SECTIONS COMPLETE — notebook ran end-to-end successfully.')
print('=' * 60)

# Final summary
print()
print('FINAL BENCHMARK SUMMARY')
print('-' * 60)
print(f'{"System":<22} {"ROUGE-L":>10} {"Token F1":>10} {"BERTScore":>12}')
print('-' * 60)
for name, r, f, b in [
    ('Base Model',       base_rouge, base_tf1['f1'], base_bscore),
    ('RAG Baseline',     rag_rouge,  rag_tf1['f1'],  rag_bscore),
    ('QLoRA Fine-tuned', ft_rouge,   ft_tf1['f1'],   ft_bscore),
]:
    print(f'{name:<22} {r:>10.4f} {f:>10.4f} {b:>12.4f}')
print('-' * 60)
print(f'{"QLoRA Δ vs RAG":<22} {ft_rouge-rag_rouge:>+10.4f} {ft_tf1["f1"]-rag_tf1["f1"]:>+10.4f} {ft_bscore-rag_bscore:>+12.4f}')
print('=' * 60)

PUSH TO HUGGINGFACE HUB

Step 1 — Authenticate (run once):
    !huggingface-cli login

Step 2 — Push adapter + tokenizer:
    model.push_to_hub("kumar-baibhav/llama3.2-3b-cuad-qlora", safe_serialization=True)
    tokenizer.push_to_hub("kumar-baibhav/llama3.2-3b-cuad-qlora")

Step 3 (optional) — Merge adapters into base for standalone model:
    merged = model.merge_and_unload()
    merged.push_to_hub("kumar-baibhav/llama3.2-3b-cuad-qlora-merged")

------------------------------------------------------------
Programmatic push (uncomment after huggingface-cli login):
------------------------------------------------------------

Adapter directory  : cuad-qlora-adapter/
Target HF repo     : https://huggingface.co/kumar-baibhav/llama3.2-3b-cuad-qlora
GPU memory (final) : 3.60 GB

ALL SECTIONS COMPLETE — notebook ran end-to-end successfully.

FINAL BENCHMARK SUMMARY
------------------------------------------------------------
System                    ROUGE-L   Token F1    BERTScore
--------